Add all the imports needed

In [ ]:
import re
import numpy as np
from pathlib import Path

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import matplotlib.pyplot as plt
import matplotlib.cm as cm

print('All imports OK')

Define the constants used

In [ ]:
# Regex to match Unicode superscript digits (⁰¹²³⁴⁵⁶⁷⁸⁹)
SUPERSCRIPT_UNICODE = r'[\u00b2\u00b3\u00b9\u2070\u2074\u2075\u2076\u2077\u2078\u2079]+'

# Matches digits glued directly after a letter — the signature of a
# superscript footnote reference merged into the word by pypdf.
# e.g. 'need85 ' or 'stage88.' → matched
# '12 March' or '2024,'      → NOT matched (space before digit)
INLINE_FOOTNOTE = r'(?<=[a-zA-Z])\d{1,3}(?=[\s\.,;:\)\-]|$)'

# Primary split boundary: newline followed by NPPF paragraph number
# e.g. '\n237. ' triggers a split
PARA_BOUNDARY = r'\n(?=\d{1,3}\. )'

# Fallback splitter settings for paragraphs over 1000 chars
FALLBACK_CHUNK_SIZE = 1000
FALLBACK_OVERLAP    = 100

# File paths — adjust if your layout differs
PDF_PATH        = Path('../data/nppf.pdf')
EMBEDDINGS_PATH = Path('../data/nppf_embeddings.npy')
CHROMA_PATH     = '../data/chroma_db'

print('Constants set')

Define the functions for extracting text from pdf and cleaning them (remove footnotes) and for chunking text up via paragraph & recursive splitter

In [ ]:
def clean_page_text(text: str) -> str:
    """
    Clean a single page of pypdf-extracted NPPF text.

    Step 1: Remove Unicode superscript digits.
    Step 2: Remove ASCII inline footnote reference numbers glued to words
            (e.g. 'need85' -> 'need'). Track which numbers are removed —
            these are the genuine footnote reference numbers for this page.
    Step 3: Remove the footnote block at the bottom of the page. A line is
            treated as the start of the footnote block only if its leading
            number was one of the tracked inline refs. This prevents false
            positives from body text like '12 December 2024'.
    Step 4: Normalise whitespace.
    """
    # Step 1: Unicode superscripts
    text = re.sub(SUPERSCRIPT_UNICODE, '', text)

    # Step 2: ASCII inline footnotes — remove and track reference numbers
    stripped_refs = set()

    def _remove_and_track(m):
        n = int(m.group())
        if n <= 120:
            stripped_refs.add(n)
            return ''
        return m.group()

    text = re.sub(INLINE_FOOTNOTE, _remove_and_track, text)

    # Step 3: Remove footnote block
    lines = text.split('\n')
    footnote_start = None

    if stripped_refs:
        for i, line in enumerate(lines):
            m = re.match(r'^(\d{1,3})\s+[A-Z]', line.strip())
            if m and int(m.group(1)) in stripped_refs:
                footnote_start = i
                break

    if footnote_start is not None:
        lines = lines[:footnote_start]

    text = '\n'.join(lines)

    # Step 4: Normalise whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = '\n'.join(line.rstrip() for line in text.split('\n'))

    return text.strip()


print('clean_page_text() defined')

def hybrid_chunk(full_text: str) -> list:
    """
    Split NPPF text using a two-level strategy:

    Primary:  split on NPPF paragraph boundaries (\n\d{1,3}. )
              Each numbered paragraph becomes its own chunk.
    Fallback: any chunk over FALLBACK_CHUNK_SIZE chars is further
              split with RecursiveCharacterTextSplitter.

    Returns a list of dicts:
        text      — the chunk string
        chunk_id  — sequential integer index
        para_num  — NPPF paragraph number if detectable, else None
    """
    fallback_splitter = RecursiveCharacterTextSplitter(
        chunk_size=FALLBACK_CHUNK_SIZE,
        chunk_overlap=FALLBACK_OVERLAP,
        separators=['\n\n', '\n', ' '],
    )

    raw_splits = re.split(PARA_BOUNDARY, full_text)

    chunks = []
    chunk_id = 0

    for split in raw_splits:
        split = split.strip()
        if not split:
            continue

        # Extract leading paragraph number for metadata if present
        para_match = re.match(r'^(\d{1,3})\. ', split)
        para_num = int(para_match.group(1)) if para_match else None

        if len(split) <= FALLBACK_CHUNK_SIZE:
            chunks.append({
                'text':     split,
                'chunk_id': chunk_id,
                'para_num': para_num,
            })
            chunk_id += 1
        else:
            # Paragraph too large — apply recursive fallback
            sub_chunks = fallback_splitter.split_text(split)
            for j, sub in enumerate(sub_chunks):
                chunks.append({
                    'text':     sub,
                    'chunk_id': chunk_id,
                    'para_num': para_num if j == 0 else None,
                })
                chunk_id += 1

    return chunks


print('hybrid_chunk() defined')

Extract the chunks from the pdf file

In [ ]:
print(f'Loading PDF from {PDF_PATH} ...')
reader = PdfReader(PDF_PATH)

cleaned_pages = []
for i, page in enumerate(reader.pages):
    raw = page.extract_text()
    if raw:
        cleaned_pages.append(clean_page_text(raw))

full_text = '\n\n'.join(cleaned_pages)
print(f'Pages loaded: {len(cleaned_pages)}')

print('Chunking ...')
chunks = hybrid_chunk(full_text)

texts  = [c['text']     for c in chunks]
ids    = [c['chunk_id'] for c in chunks]
paras  = [c['para_num'] for c in chunks]

sizes = [len(t) for t in texts]
print(f'Total chunks : {len(chunks)}')
print(f'Char lengths  — mean: {np.mean(sizes):.0f}, '
      f'median: {np.median(sizes):.0f}, '
      f'max: {max(sizes)}, min: {min(sizes)}')

# Spot check — print paragraph 237 to verify cleaning is correct
target = next((c for c in chunks if c['para_num'] == 237), None)
if target:
    print(f'\n--- Paragraph 237 (spot check) ---')
    print(target['text'])
else:
    print('Paragraph 237 not found as a standalone chunk (may have been merged)')

Use embedding model to create 384-dim vectors from each chunk - use sentence transformer library and all-MiniLM-L6-v2 as lightweight model. Use directly on CPU rather than using ollama as it is quicker and less encoder models available on ollama.

In [ ]:


if EMBEDDINGS_PATH.exists():
    print(f'Loading existing embeddings from {EMBEDDINGS_PATH}')
    embeddings = np.load(EMBEDDINGS_PATH)
else:
    print('Loading embedding model (downloads ~90MB on first run) ...')
    model = SentenceTransformer('all-MiniLM-L6-v2')

    print(f'Embedding {len(texts)} chunks — takes ~30-60s on CPU ...')
    embeddings = model.encode(
        texts,
        show_progress_bar=True,
        convert_to_numpy=True,
        batch_size=32,
    )

    EMBEDDINGS_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.save(EMBEDDINGS_PATH, embeddings)
    print(f'Embeddings saved to {EMBEDDINGS_PATH}')

print(f'Embedding matrix shape: {embeddings.shape}')
# Expected: (n_chunks, 384)

In [ ]:
def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Find two chunks both about affordable housing, and one about minerals
# These are used to check the embedding space makes semantic sense
afi = next(i for i, t in enumerate(texts) if 'affordable housing' in t.lower())
afj = next(i for i, t in enumerate(texts)
           if 'affordable housing' in t.lower() and i != afi)
unrelated = next(i for i, t in enumerate(texts) if 'minerals' in t.lower())

sim_related   = cosine_sim(embeddings[afi], embeddings[afj])
sim_unrelated = cosine_sim(embeddings[afi], embeddings[unrelated])

print('Cosine similarity sanity check:')
print(f'  Two affordable housing chunks  ({afi} & {afj}): {sim_related:.4f}')
print(f'  Affordable housing vs minerals ({afi} & {unrelated}): {sim_unrelated:.4f}')
print()
print('The first value should be noticeably higher than the second.')
print('If both are very similar, something is wrong with the embeddings.')

Define a cosine similarity checking function - result is that two chunks with the same words in are more similar (0.6713) and with different (0.4703)

Then create a vector store using ChromaDB

In [ ]:
print(f'Initialising ChromaDB at {CHROMA_PATH} ...')
client = chromadb.PersistentClient(path=CHROMA_PATH)

# Delete and recreate so this cell is safe to re-run
try:
    client.delete_collection('nppf')
    print('  Deleted existing nppf collection')
except Exception:
    pass

collection = client.create_collection(
    name='nppf',
    metadata={'hnsw:space': 'cosine'},
)

collection.add(
    ids       = [str(i) for i in ids],
    embeddings= embeddings.tolist(),
    documents = texts,
    metadatas = [
        {
            'chunk_id': c['chunk_id'],
            'para_num': c['para_num'] if c['para_num'] is not None else -1,
        }
        for c in chunks
    ],
)

print(f'Ingested {collection.count()} chunks into ChromaDB')

In [ ]:
# Load the model if not already in memory from Cell 6
# (If you ran Cell 6 and the embeddings already existed on disk,
#  the model variable won't be set — this handles that case.)
try:
    model
except NameError:
    print('Loading embedding model ...')
    model = SentenceTransformer('all-MiniLM-L6-v2')

TEST_QUERIES = [
    "What is the presumption in favour of sustainable development?",
    "What are the affordable housing requirements for Green Belt land?",
    "How should local planning authorities assess flood risk?",
]

print('=' * 60)
for query in TEST_QUERIES:
    q_embedding = model.encode([query], convert_to_numpy=True)
    results = collection.query(
        query_embeddings=q_embedding.tolist(),
        n_results=3,
        include=['documents', 'distances', 'metadatas'],
    )
    print(f'\nQuery: "{query}"')
    for i, (doc, dist, meta) in enumerate(zip(
        results['documents'][0],
        results['distances'][0],
        results['metadatas'][0],
    )):
        para_label = f"para {meta['para_num']}" if meta['para_num'] != -1 else 'no para'
        sim = 1 - dist  # ChromaDB returns distance; convert to similarity
        print(f'  [{i+1}] sim={sim:.4f} | {para_label} | {doc[:120].replace(chr(10), " ")}...')
print('=' * 60)

Use some test queries and see what the cosine similarities are for each of the retrieved chunks and if they are actually in the right ballpark. Scores all > 0.6 and from initial inspection seem to be relevant to the query. But will need to test further when using the LLM to answer some queries.

In [ ]:
K = 15  # first guess — NPPF has 17 chapters so 15 clusters is reasonable

print(f'Running K-Means with k={K} in 384-dimensional embedding space ...')
print('(k-means++ initialisation, 10 restarts — takes ~10-20s)')

kmeans = KMeans(
    n_clusters=K,
    init='k-means++',
    n_init=10,
    random_state=42,
)
cluster_labels = kmeans.fit_predict(embeddings)

inertia   = kmeans.inertia_
sil_score = silhouette_score(
    embeddings, cluster_labels, sample_size=200, random_state=42
)

print(f'\nK-Means results (k={K}):')
print(f'  Inertia (J):      {inertia:.1f}  — lower means tighter clusters')
print(f'  Silhouette score: {sil_score:.4f} — range -1 to 1; >0.05 is reasonable for text')

# For each cluster, show size and the chunk closest to the centroid
print(f'\nCluster summary:')
for k in range(K):
    mask      = cluster_labels == k
    centroid  = kmeans.cluster_centers_[k]
    dists     = np.linalg.norm(embeddings[mask] - centroid, axis=1)
    rep_idx   = np.where(mask)[0][dists.argmin()]
    sample    = texts[rep_idx][:120].replace('\n', ' ')
    print(f'  Cluster {k:2d}: {mask.sum():3d} chunks | {sample}...')

15 clusters - between 7 and 38 chunks per cluster. Plotted the inertia below and 15 is a reasonable number in this case.

In [ ]:
# Two-stage dimensionality reduction:
#   384D --[PCA]--> 50D --[t-SNE]--> 2D
#
# PCA first is standard practice: it speeds up t-SNE significantly
# and removes noisy low-variance dimensions without losing structure.
# t-SNE then finds a 2D layout that preserves local neighbourhoods.
# Points close in 2D were close in 384D — i.e. semantically similar.

# Stage 1: PCA 384 -> 50
print('PCA: 384 → 50 dimensions ...')
pca = PCA(n_components=50, random_state=42)
embeddings_50d = pca.fit_transform(embeddings)
explained = pca.explained_variance_ratio_.sum()
print(f'  Variance retained by 50 components: {explained:.1%}')

# Stage 2: t-SNE 50 -> 2
print('t-SNE: 50 → 2 dimensions (takes ~10-30s) ...')
tsne = TSNE(
    n_components=2,
    perplexity=30,        # good default for ~300 points
    max_iter=1000,
    random_state=42,
    init='pca',           # more stable than random initialisation
    learning_rate='auto',
)
embeddings_2d = tsne.fit_transform(embeddings_50d)
print(f'  KL divergence: {tsne.kl_divergence_:.4f}  (lower = better fit)')

# Plot
fig, ax = plt.subplots(figsize=(13, 9))
colours = cm.tab20(np.linspace(0, 1, K))

for k in range(K):
    mask = cluster_labels == k
    ax.scatter(
        embeddings_2d[mask, 0],
        embeddings_2d[mask, 1],
        c=[colours[k]],
        label=f'Cluster {k} (n={mask.sum()})',
        s=45,
        alpha=0.85,
        edgecolors='white',
        linewidths=0.3,
    )

# Annotate every 10th paragraph number so you can orient yourself
for i, (x, y) in enumerate(embeddings_2d):
    if paras[i] is not None and paras[i] % 10 == 0:
        ax.annotate(
            f'§{paras[i]}',
            (x, y),
            fontsize=6,
            alpha=0.55,
            ha='center',
        )

ax.set_title(
    'NPPF chunks — t-SNE 2D projection coloured by K-Means cluster (k=15)\n'
    'Points close together in 2D were close in 384-dimensional embedding space',
    fontsize=11,
)
ax.set_xlabel('t-SNE dimension 1')
ax.set_ylabel('t-SNE dimension 2')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, framealpha=0.5)
ax.grid(True, alpha=0.2)

plt.tight_layout()

PLOT_PATH = Path('../data/tsne_clusters.png')
plt.savefig(PLOT_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved to {PLOT_PATH}')

t-SNE plot the data (first reduced by PCA from 384 --> 50 dims) with the 15-clusters identified by k-means - shows a number of distinct clusters. But also quite a lot of data in the middle of the plot which isn't that clustered - but this is common for a policy document with quite a lot of similar text throughout.

Now test the intertia plot to see whether 15 is about the right number of clusters

In [ ]:
# ── Elbow method — find optimal k ────────────────────────────
# Plot inertia J against k. Look for the 'elbow' where the curve
# flattens — adding more clusters beyond that point gives diminishing
# returns in terms of cluster tightness.

K_RANGE = range(5, 31)
inertias = []

print('Running K-Means for k=5 to 30 ...')
for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=5, random_state=42)
    km.fit(embeddings)
    inertias.append(km.inertia_)
    print(f'  k={k:2d}  J={km.inertia_:.1f}')

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(list(K_RANGE), inertias, marker='o', linewidth=2, markersize=5)
ax.axvline(x=15, color='red', linestyle='--', alpha=0.5, label='Current k=15')
ax.set_xlabel('Number of clusters k')
ax.set_ylabel('Inertia J (within-cluster sum of squares)')
ax.set_title('Elbow method — NPPF chunk embeddings')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/elbow_plot.png', dpi=150)
plt.show()
print('Elbow plot saved to data/elbow_plot.png')

The semantic space of the text embeddings is contiuous and high dimensional - chunks relatively smoothly distributed, so no levelling off of elbow. No "true" k based on the plot - so going with 15 is a reasonable number.

In [ ]:
# ── Cluster labelling ─────────────────────────────────────────
# For each cluster, print the representative chunk (closest to
# centroid) so you can manually assign a topic label.
# This is the interpretability check that matters for the portfolio.

from sklearn.feature_extraction.text import TfidfVectorizer

# Fit a TF-IDF vectoriser to find the most distinctive terms per cluster
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(texts)
feature_names = np.array(tfidf.get_feature_names_out())

print(f'Cluster labels for k={K}')
print('=' * 70)

for k in range(K):
    mask = cluster_labels == k

    # Representative chunk — closest to centroid in embedding space
    centroid = kmeans.cluster_centers_[k]
    dists = np.linalg.norm(embeddings[mask] - centroid, axis=1)
    rep_idx = np.where(mask)[0][dists.argmin()]

    # Top TF-IDF terms for this cluster
    cluster_tfidf = tfidf_matrix[mask].mean(axis=0)
    cluster_tfidf = np.asarray(cluster_tfidf).flatten()
    top_term_idx = cluster_tfidf.argsort()[-5:][::-1]
    top_terms = ', '.join(feature_names[top_term_idx])

    print(f'\nCluster {k:2d} (n={mask.sum():3d})')
    print(f'  Top terms:   {top_terms}')
    print(f'  Rep chunk:   {texts[rep_idx][:200].replace(chr(10), " ")}')
    print('-' * 70)

Based on tf-idf for each cluster and the cluster closest to the centroid, labels for each cluster are:
Clusters labels:
Cluster 0: Historic
Cluster 1: Housing delivery
Cluster 2: Towns & Cities
Cluster 3: Affordability
Cluster 4: Climate
Cluster 5: Flood Risk
CLuster 6: Land use
Cluster 7: Planning regulations
CLuster 8: Decisions
Cluster 9: Nature
Cluster 10: Planning framework
Cluster 11: Transport
Cluster 12: Minerals
Cluster 13: Green belt
